In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin
from sklearn.utils import check_X_y, check_array
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error



# Helper: target statistics for categorical features (ordered)

def compute_cat_features(X_cat, y, prior, smoothing, random_state=42):
    """
    Convert categorical features to numeric using ordered target statistics.
    For each categorical column, compute smoothed mean target for each category
    using only previous samples (to avoid leakage). 
    Returns: array of encoded features (same shape as X_cat).
    """
    X_cat = np.asarray(X_cat)
    n_samples, n_cats = X_cat.shape
    y = np.asarray(y)
    encoded = np.zeros_like(X_cat, dtype=np.float64)

    for col in range(n_cats):
        perm = np.random.RandomState(random_state + col).permutation(n_samples)
        counts = {}
        sums = {}
        prior_val = prior

        for idx in perm:
            cat = X_cat[idx, col]
            if cat in sums:
                total_sum = sums[cat]
                total_count = counts[cat]
                encoded[idx, col] = (total_sum + prior_val * smoothing) / (total_count + smoothing)
            else:
                encoded[idx, col] = prior_val

            if cat in sums:
                sums[cat] += y[idx]
                counts[cat] += 1
            else:
                sums[cat] = y[idx]
                counts[cat] = 1

    return encoded



# Symmetric (oblivious) decision tree for CatBoost (CORRECTED)

class SymmetricTree:
    """
    A symmetric tree where all nodes at the same depth share the same feature and threshold.
    Stores splits and leaf weights in a dictionary keyed by node index (binary path).
    """
    def __init__(self, max_depth=6, min_data_in_leaf=5, lambda_=1.0, gamma=0.0):
        self.max_depth = max_depth
        self.min_data_in_leaf = min_data_in_leaf
        self.lambda_ = lambda_
        self.gamma = gamma
        self.splits = []                     # list of (feature_idx, threshold) per depth
        self.leaf_weight_dict = {}           # node_idx -> leaf weight

    def _compute_gain(self, G, H, G_L, H_L, G_R, H_R):
        def term(g, h):
            return g**2 / (h + self.lambda_)
        gain = 0.5 * (term(G_L, H_L) + term(G_R, H_R) - term(G_L + G_R, H_L + H_R)) - self.gamma
        return gain

    def fit(self, X_binned, G, H, bin_edges):
        """
        Build a symmetric tree using level‑wise growth.
        X_binned: shape (n_samples, n_features) with integer bin indices.
        bin_edges: list of arrays for each feature (used only to get number of bins)
        """
        n_samples = X_binned.shape[0]
        n_features = X_binned.shape[1]

        # Each leaf is (samples, node_idx). Start with root node 0.
        leaves = [(np.arange(n_samples, dtype=np.int64), 0)]
        self.splits = []

        for depth in range(self.max_depth):
            best_gain_total = -np.inf
            best_feature = None
            best_threshold = None
            best_new_leaves = None  # list of (left_samples, right_samples, left_node, right_node)

            # Try all features
            for f in range(n_features):
                n_bins = len(bin_edges[f]) - 1
                if n_bins < 2:
                    continue

                # Precompute histograms for each leaf for this feature
                leaf_data = []  # list of (leaf_samples, hist_G, hist_H, total_G, total_H, node_idx)
                for leaf_samples, node_idx in leaves:
                    if len(leaf_samples) < 2 * self.min_data_in_leaf:
                        # skip splitting this leaf
                        leaf_data.append((leaf_samples, None, None, None, None, node_idx))
                        continue
                    bin_ids = X_binned[leaf_samples, f].astype(np.int32)
                    G_sub = G[leaf_samples]
                    H_sub = H[leaf_samples]
                    hist_G = np.zeros(n_bins, dtype=np.float64)
                    hist_H = np.zeros(n_bins, dtype=np.float64)
                    np.add.at(hist_G, bin_ids, G_sub)
                    np.add.at(hist_H, bin_ids, H_sub)
                    total_G = np.sum(G_sub)
                    total_H = np.sum(H_sub)
                    leaf_data.append((leaf_samples, hist_G, hist_H, total_G, total_H, node_idx))

                # Evaluate each threshold t (split between bin t and t+1)
                for t in range(n_bins - 1):
                    total_gain = 0.0
                    valid = True
                    new_leaves_for_this_threshold = []
                    for leaf_samples, hist_G, hist_H, total_G, total_H, node_idx in leaf_data:
                        if hist_G is None:
                            # can't split this leaf, keep as is
                            new_leaves_for_this_threshold.append((leaf_samples, None, node_idx, None))
                            continue
                        # left: bins <= t, right: > t
                        G_L = np.sum(hist_G[:t+1])
                        H_L = np.sum(hist_H[:t+1])
                        G_R = total_G - G_L
                        H_R = total_H - H_L
                        if H_L < self.min_data_in_leaf or H_R < self.min_data_in_leaf:
                            valid = False
                            break
                        gain = self._compute_gain(total_G, total_H, G_L, H_L, G_R, H_R)
                        total_gain += gain  # sum gains (they may be negative)
                    if not valid:
                        continue
                    if total_gain > best_gain_total:
                        best_gain_total = total_gain
                        best_feature = f
                        best_threshold = t
                        # Build the new leaves for this split
                        new_leaves = []
                        for leaf_samples, hist_G, hist_H, total_G, total_H, node_idx in leaf_data:
                            if hist_G is None:
                                new_leaves.append((leaf_samples, None, node_idx, None))
                            else:
                                bin_ids = X_binned[leaf_samples, f]
                                left_mask = (bin_ids <= t)
                                right_mask = ~left_mask
                                left_samples = leaf_samples[left_mask]
                                right_samples = leaf_samples[right_mask]
                                if len(left_samples) < self.min_data_in_leaf or len(right_samples) < self.min_data_in_leaf:
                                    new_leaves.append((leaf_samples, None, node_idx, None))
                                else:
                                    new_leaves.append((left_samples, right_samples, node_idx*2, node_idx*2+1))
                        best_new_leaves = new_leaves

            if best_feature is None or best_gain_total <= 0:
                break

            # Apply split
            self.splits.append((best_feature, best_threshold))
            new_leaves_list = []
            for left_samples, right_samples, left_node, right_node in best_new_leaves:
                if right_samples is not None:
                    new_leaves_list.append((left_samples, left_node))
                    new_leaves_list.append((right_samples, right_node))
                else:
                    new_leaves_list.append((left_samples, left_node))
            leaves = new_leaves_list

        # Compute leaf weights for each final leaf and store in dict
        self.leaf_weight_dict = {}
        for leaf_samples, node_idx in leaves:
            G_leaf = np.sum(G[leaf_samples])
            H_leaf = np.sum(H[leaf_samples])
            weight = -G_leaf / (H_leaf + self.lambda_)
            self.leaf_weight_dict[node_idx] = weight

    def predict(self, X_binned):
        """Return leaf weights for each sample."""
        n_samples = X_binned.shape[0]
        preds = np.zeros(n_samples)
        for i in range(n_samples):
            node_idx = 0
            for feature, threshold in self.splits:
                if X_binned[i, feature] <= threshold:
                    node_idx = node_idx * 2      # left
                else:
                    node_idx = node_idx * 2 + 1  # right
            preds[i] = self.leaf_weight_dict.get(node_idx, 0.0)
        return preds



# CatBoost Base Booster

class CatBoostBase:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=6,
                 min_data_in_leaf=5, lambda_=1.0, gamma=0.0,
                 cat_features=None, prior=0.5, smoothing=1.0,
                 subsample=1.0, early_stopping_rounds=None, random_state=None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_data_in_leaf = min_data_in_leaf
        self.lambda_ = lambda_
        self.gamma = gamma
        self.cat_features = cat_features  # list of indices for categorical columns
        self.prior = prior
        self.smoothing = smoothing
        self.subsample = subsample
        self.early_stopping_rounds = early_stopping_rounds
        self.random_state = random_state
        self.trees = []
        self.initial_pred = None
        self.bin_edges = None

    def _init_prediction(self, y):
        raise NotImplementedError

    def _get_gradient_hessian(self, y, pred):
        raise NotImplementedError

    def _loss(self, y, pred):
        raise NotImplementedError

    def _preprocess_categorical(self, X, y_fit=None):
        """
        Encode categorical features using ordered target statistics.
        If y_fit is provided, use it to compute TS; otherwise, use stored mapping.
        For simplicity, this implementation does not store the mapping from
        fit – so prediction with categorical features is not supported.
        """
        if self.cat_features is None or len(self.cat_features) == 0:
            return X
        X = np.asarray(X)
        X_cat = X[:, self.cat_features]
        if y_fit is not None:
            encoded = compute_cat_features(X_cat, y_fit, self.prior, self.smoothing, self.random_state)
        else:
            raise ValueError("Categorical features need to be encoded during fit; "
                             "this simplified implementation does not store the mapping for prediction.")
        X_processed = np.copy(X).astype(np.float64)
        X_processed[:, self.cat_features] = encoded
        return X_processed

    def _discretize_features(self, X, num_bins=255):
        """Create bin edges and binned data for all features."""
        n_features = X.shape[1]
        bin_edges = []
        X_binned = np.zeros_like(X, dtype=np.int32)
        for f in range(n_features):
            col = X[:, f]
            quantiles = np.linspace(0, 100, num_bins + 1)
            edges = np.percentile(col, quantiles)
            edges = np.unique(edges)
            bin_edges.append(edges)
            X_binned[:, f] = np.digitize(col, edges[:-1]) - 1
            max_idx = len(edges) - 2
            X_binned[:, f] = np.clip(X_binned[:, f], 0, max_idx)
        return X_binned, bin_edges

    def fit(self, X, y, X_val=None, y_val=None):
        X, y = check_X_y(X, y)
        np.random.seed(self.random_state)
        self.trees = []
        n_samples = X.shape[0]

        # Preprocess categorical features (if any)
        if self.cat_features is not None:
            X = self._preprocess_categorical(X, y)

        # Initial prediction
        self.initial_pred = self._init_prediction(y)
        pred = np.full(n_samples, self.initial_pred)

        # Binning for all features (for histogram-based splits)
        self.X_binned, self.bin_edges = self._discretize_features(X)

        if X_val is not None:
            if self.cat_features is not None:
                # For simplicity, we do not handle categorical validation data.
                # In practice, you'd need to apply the same encoding.
                # We'll just raise a warning and proceed assuming no categorical.
                print("Warning: Categorical features in validation set are not handled.")
            X_val_binned = self._apply_binning(X_val, self.bin_edges)
            pred_val = np.full(X_val.shape[0], self.initial_pred)

        best_val_loss = np.inf
        no_improve = 0

        for i in range(self.n_estimators):
            # Subsample
            if self.subsample < 1.0:
                idx = np.random.choice(n_samples, int(n_samples * self.subsample), replace=False)
                X_sub = self.X_binned[idx]
                y_sub = y[idx]
                pred_sub = pred[idx]
            else:
                X_sub = self.X_binned
                y_sub = y
                pred_sub = pred

            G, H = self._get_gradient_hessian(y_sub, pred_sub)

            tree = SymmetricTree(
                max_depth=self.max_depth,
                min_data_in_leaf=self.min_data_in_leaf,
                lambda_=self.lambda_,
                gamma=self.gamma
            )
            tree.fit(X_sub, G, H, self.bin_edges)

            pred += self.learning_rate * tree.predict(self.X_binned)
            self.trees.append(tree)

            if X_val is not None:
                pred_val += self.learning_rate * tree.predict(X_val_binned)
                val_loss = self._loss(y_val, pred_val)
                if val_loss < best_val_loss - 1e-7:
                    best_val_loss = val_loss
                    no_improve = 0
                else:
                    no_improve += 1
                if self.early_stopping_rounds is not None and no_improve >= self.early_stopping_rounds:
                    self.trees = self.trees[:i+1 - self.early_stopping_rounds]
                    break

        return self

    def _apply_binning(self, X, bin_edges):
        """Apply precomputed bin edges to new data."""
        X_binned = np.zeros_like(X, dtype=np.int32)
        for f in range(X.shape[1]):
            edges = bin_edges[f]
            X_binned[:, f] = np.digitize(X[:, f], edges[:-1]) - 1
            max_idx = len(edges) - 2
            X_binned[:, f] = np.clip(X_binned[:, f], 0, max_idx)
        return X_binned

    def predict_raw(self, X):
        X = check_array(X)
        if self.cat_features is not None:
            # This simplified implementation does not handle categorical features in prediction.
            # If you use cat_features, you must ensure X is already encoded.
            # We'll just issue a warning if cat_features is set.
            print("Warning: Categorical features in prediction are not automatically encoded.")
        X_binned = self._apply_binning(X, self.bin_edges)
        pred = np.full(X.shape[0], self.initial_pred)
        for tree in self.trees:
            pred += self.learning_rate * tree.predict(X_binned)
        return pred



# Regressor and Classifier

class CatBoostRegressor(CatBoostBase, RegressorMixin):
    def _init_prediction(self, y):
        return np.mean(y)

    def _get_gradient_hessian(self, y, pred):
        G = pred - y
        H = np.ones_like(y)
        return G, H

    def _loss(self, y, pred):
        return np.mean((y - pred) ** 2)

    def predict(self, X):
        return self.predict_raw(X)


class CatBoostClassifier(CatBoostBase, ClassifierMixin):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.classes_ = None

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-x))

    def _init_prediction(self, y):
        self.classes_ = np.unique(y)
        p_pos = np.mean(y == self.classes_[1])
        p_pos = np.clip(p_pos, 1e-15, 1 - 1e-15)
        return np.log(p_pos / (1 - p_pos))

    def _get_gradient_hessian(self, y, pred):
        p = self._sigmoid(pred)
        G = p - y
        H = p * (1 - p)
        return G, H

    def _loss(self, y, pred):
        p = self._sigmoid(pred)
        p = np.clip(p, 1e-15, 1 - 1e-15)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

    def predict_proba(self, X):
        raw = self.predict_raw(X)
        p_pos = self._sigmoid(raw)
        p_pos = np.clip(p_pos, 1e-15, 1 - 1e-15)
        return np.column_stack((1 - p_pos, p_pos))

    def predict(self, X):
        proba = self.predict_proba(X)
        return np.where(proba[:, 1] >= 0.5, self.classes_[1], self.classes_[0])



# Example usage

if __name__ == "__main__":
    # ----- Regression -----
    print("--- Regression ---")
    X, y = make_regression(n_samples=300, n_features=5, noise=10, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    cat_reg = CatBoostRegressor(
        n_estimators=50, learning_rate=0.1, max_depth=4,
        min_data_in_leaf=3, lambda_=0.1, gamma=0.0,
        subsample=0.8, random_state=42
    )
    cat_reg.fit(X_train, y_train)
    y_pred = cat_reg.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Test MSE: {mse:.3f}")

    # ----- Binary Classification -----
    print("\n--- Classification ---")
    X, y = make_classification(n_samples=300, n_features=10, n_informative=8,
                               n_redundant=2, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    cat_clf = CatBoostClassifier(
        n_estimators=50, learning_rate=0.1, max_depth=4,
        min_data_in_leaf=3, lambda_=0.1, gamma=0.0,
        subsample=0.8, random_state=42,
        early_stopping_rounds=10
    )
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
    cat_clf.fit(X_train, y_train, X_val=X_val, y_val=y_val)
    y_pred = cat_clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Test accuracy: {acc:.3f}")

    proba = cat_clf.predict_proba(X_test[:5])
    print("Probabilities (first 5):\n", proba)

--- Regression ---
Test MSE: 1446.820

--- Classification ---
Test accuracy: 0.789
Probabilities (first 5):
 [[0.38706916 0.61293084]
 [0.70640045 0.29359955]
 [0.55338872 0.44661128]
 [0.65332658 0.34667342]
 [0.77866909 0.22133091]]
